In [1]:
import pingouin

import sys
sys.path.append('../mdi/')
import globals as gl

import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import os
import pandas as pd

plt.style.use('default')

base directory: /cifs/diedrichsen/data/ModifiedDigitInterference/


In [2]:
data = pd.read_csv('/home/alily/Documents/GitHub/modified-digit-inteference/mdi/MDI0_merged1.csv')
N = data.SID.nunique()
data = data[(data.correct==1) & (data.BN>1)]
data[['ipi1', 'ipi2', 'ipi3', 'ipi4']] = data[['ipi1', 'ipi2', 'ipi3', 'ipi4']].astype(float)
data.PosInQuartet = pd.Categorical(data.PosInQuartet, categories=[1, 2, 3, 4], ordered=True)
data.Quartet = pd.Categorical(data.Quartet, categories=['AAAA', 'AAMA', 'AARA'], ordered=True)

data.head(6)

,BN,TN,startTR,startTRReal,startTimeReal,planTime,execTime,feedbackTime,iti,expectedDigit1,...,SN,SID,PosInQuartet,Quartet,planError,correct,ipi1,ipi2,ipi3,ipi4
74,2,3,0,15,14471,1000,5000,1000,500,5,...,0,100,3,AAMA,False,True,945.0,700.0,325.0,320.0
75,2,4,0,20,19606,1000,5000,1000,500,5,...,0,100,4,AAMA,False,True,550.0,410.0,365.0,265.0
76,2,5,0,25,24666,1000,5000,1000,500,2,...,0,100,1,AARA,False,True,370.0,425.0,395.0,385.0
79,2,8,0,43,42281,1000,5000,1000,500,2,...,0,100,4,AARA,False,True,825.0,560.0,505.0,555.0
80,2,9,0,48,47901,1000,5000,1000,500,4,...,0,100,1,AARA,False,True,685.0,940.0,570.0,650.0
82,2,11,0,60,59621,1000,5000,1000,500,4,...,0,100,3,AARA,False,True,540.0,640.0,255.0,80.0


In [3]:
condition_data = data[(data.Quartet!="AAAA")]
condition_data.groupby(["SID","Quartet","PosInQuartet"])

anova=pingouin.rm_anova(data=condition_data,dv="MovementTime",subject="SN",within=["Quartet","PosInQuartet"])
display(anova)

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,Quartet,3569.512707,1,19,3569.512707,0.324488,5.755970e-01,5.755970e-01,0.000124,1.000000
1,PosInQuartet,337618.007494,3,57,112539.335831,20.717528,3.344193e-09,4.387763e-09,0.011606,0.983493
2,Quartet * PosInQuartet,15179.961339,3,57,5059.987113,0.889662,4.520789e-01,4.353033e-01,0.000528,0.810648


In [4]:
melted_ipi = condition_data.melt(id_vars=["Quartet", 'SID', 'PosInQuartet', 'TN', 'BN'],value_vars=["ipi1","ipi2","ipi3","ipi4"], value_name="IPI", var_name="IPI_id")
melted_ipi['IPI'] = (melted_ipi['IPI'] - melted_ipi.groupby('SID')['IPI'].transform('mean'))
melted_ipiG = melted_ipi.groupby(['SID', 'Quartet', 'PosInQuartet', 'IPI_id'], observed=True).mean(numeric_only=True).reset_index()
melted_ipiG['IPI'] += melted_ipiG['IPI'].mean()



In [5]:
melted_ipi.head(6)

,Quartet,SID,PosInQuartet,TN,BN,IPI_id,IPI
0,AAMA,100,3,3,2,ipi1,548.738789
1,AAMA,100,4,4,2,ipi1,153.738789
2,AARA,100,1,5,2,ipi1,-26.261211
3,AARA,100,4,8,2,ipi1,428.738789
4,AARA,100,1,9,2,ipi1,288.738789
5,AARA,100,3,11,2,ipi1,143.738789


In [ ]:
#ipi_anova = AnovaRM(data=melted_ipi,subject="SID",depvar="IPI",within=["PosInQuartet","IPI_id"],aggregate_func="mean")

filtered_melted_ipi = melted_ipi[melted_ipi["Quartet"] != "AAAA"]

filtered_melted_ipi3 = filtered_melted_ipi[melted_ipi["PosInQuartet"] == 3] #trials that are the 3rd in the quartet
filtered_melted_ipi4 = filtered_melted_ipi[melted_ipi["PosInQuartet"] == 4] #every 4th trial in the quartet

ipi_anova=pingouin.rm_anova(data=filtered_melted_ipi3,dv="IPI",subject="SID",within=["IPI_id","Quartet"]) #anova on every 3rd trial


display(ipi_anova)

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,IPI_id,118475.036126,3,57,39491.678709,17.691412,3.084754e-08,3.399273e-07,0.392460,0.832593
1,Quartet,677.159296,1,19,677.159296,1.218252,2.834873e-01,2.834873e-01,0.003679,1.000000
2,IPI_id * Quartet,5725.407804,3,57,1908.469268,2.912976,4.204595e-02,5.336678e-02,0.030273,0.826275


In [ ]:
ipi_anova2=pingouin.rm_anova(data=filtered_melted_ipi4,dv="IPI",subject="SID",within=["IPI_id","Quartet"]) #anova on the last trial on every quartet


display(ipi_anova2)

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,ng2,eps
0,IPI_id,69238.196376,3,57,23079.398792,14.681551,3.374859e-07,0.000023,0.364223,0.651346
1,Quartet,3853.111768,1,19,3853.111768,6.791289,1.736106e-02,0.017361,0.030896,1.000000
2,IPI_id * Quartet,827.198393,3,57,275.732798,1.037133,3.831150e-01,0.371526,0.006798,0.770215
